# Lekcja 2.2 — eksperymenty: BFS i DFS w magazynie

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz wykonać eksperymenty przed skończeniem zadania.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.search import reference as algorithms

Funkcja `run` uruchamia jedno przeszukiwanie na mapie i zwraca węzeł celu oraz listę rozwiniętych pól. Funkcja `summary` wypisuje długość trasy i liczbę rozwinięć. Na wykresach niebieskie pola to pola rozwinięte, a pomarańczowe to trasa.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from introai.trace import TraceLimitExceeded, record
from introai.world.plot import plot_map
from introai.world.warehouse import RouteProblem, WarehouseMap, render_text

MAPS = Path.cwd().parents[2] / "data" / "maps"


def run(algorithm, warehouse, action_order=("N", "E", "S", "W"), limit=100_000):
    """Run one search on a map; return the goal node and the expanded cells."""
    try:
        with record(limit=limit) as trace:
            node = algorithm(RouteProblem(warehouse, action_order))
    except TraceLimitExceeded:
        raise RuntimeError(
            f"{algorithm.__name__} expanded more than {limit} nodes - check task.py "
            '(or set IMPLEMENTATION = "reference" in the first cell)'
        ) from None
    expanded = [event.data["node"].state for event in trace.of_kind("expand")]
    return node, expanded


def summary(label, node, expanded):
    steps = len(node.solution()) if node else None
    print(f"{label}: route length {steps}, expanded {len(expanded)}")

## Eksperyment 1: mapa z alejkami

Robot `R` stoi w górnym korytarzu, a rampa `D` leży w lewym dolnym rogu. Robot próbuje ruchów w kolejności N, E, S, W.

In [ ]:
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
print(render_text(aisles))

**Twoje przewidywanie.** Która metoda znajdzie krótszą trasę? Która rozwinie więcej pól? Mniej więcej ile?

*Moje przewidywanie:* …

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, ["bfs", "dfs"]):
    node, expanded = run(getattr(algorithms, name), aisles)
    summary(name, node, expanded)
    path = node.states() if node else []
    plot_map(aisles, path=path, expanded=expanded, title=name.upper(), ax=ax)
fig.tight_layout()

**Do przemyślenia.** Porównaj wynik ze swoim przewidywaniem. Dlaczego DFS ruszył na wschód, skoro rampa jest na zachodzie? Czy DFS w ogóle „próbuje” znaleźć krótką trasę?

## Eksperyment 2: kolejność akcji

Teraz ta sama mapa, ale robot próbuje ruchów w kolejności W, S, E, N.

**Twoje przewidywanie.** Jak zmieni się trasa BFS, a jak trasa DFS?

*Moje przewidywanie:* …

In [ ]:
for order in [("N", "E", "S", "W"), ("W", "S", "E", "N")]:
    for name in ["bfs", "dfs"]:
        node, expanded = run(getattr(algorithms, name), aisles, order)
        summary(f"{name}, order {' '.join(order)}", node, expanded)

In [ ]:
node, expanded = run(algorithms.dfs, aisles, ("W", "S", "E", "N"))
path = node.states() if node else []
ax = plot_map(aisles, path=path, expanded=expanded, title="DFS, order W S E N")

**Do przemyślenia.** Która metoda jest wrażliwa na kolejność akcji, a która nie? Czy dobry wynik DFS przy tej kolejności znaczy, że DFS dobrze nadaje się do tego problemu?

## Eksperyment 3: otwarta przestrzeń

Na mapie `open_floor` prawie nie ma przeszkód, a rampa leży blisko startu.

**Twoje przewidywanie.** Która metoda rozwinie więcej pól niż w alejkach? Jak długa będzie trasa DFS?

*Moje przewidywanie:* …

In [ ]:
open_floor = WarehouseMap.from_file(MAPS / "open_floor.txt")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, ["bfs", "dfs"]):
    node, expanded = run(getattr(algorithms, name), open_floor)
    summary(name, node, expanded)
    path = node.states() if node else []
    plot_map(open_floor, path=path, expanded=expanded, title=name.upper(), ax=ax)
fig.tight_layout()

**Do przemyślenia.** Dlaczego BFS rozwija tu tak wiele pól, choć rampa jest blisko? Dlaczego trasa DFS jest tu jeszcze dłuższa niż w alejkach? Jaka informacja pomogłaby obu algorytmom? Odpowiedź pojawi się w lekcji 2.4.

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/02_bfs_dfs/answers.pl.md`.